In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, learning_curve, validation_curve
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, AdaBoostRegressor, ExtraTreesRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.svm import SVR
import xgboost as xgb
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error, median_absolute_error
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Load the dataset
df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Final_Dataset_FR.csv')

# Convert time to datetime format
df['time'] = pd.to_datetime(df['time'], format='ISO8601')

# Extract temporal features from time column
df['hour'] = df['time'].dt.hour
df['day_of_week'] = df['time'].dt.dayofweek
df['day_of_month'] = df['time'].dt.day
df['month'] = df['time'].dt.month
df['year'] = df['time'].dt.year

# List of categorical columns to encode
categorical_columns = ['road_type', 'weather_condition', 'road_condition', 'time_of_day']

# Initialize the OneHotEncoder
encoder = OneHotEncoder(sparse_output=False, drop='first')

# Fit and transform the categorical columns
encoded_columns = encoder.fit_transform(df[categorical_columns])

# Get the new column names
encoded_column_names = encoder.get_feature_names_out(categorical_columns)

# Create a DataFrame with the encoded columns
encoded_df = pd.DataFrame(encoded_columns, columns=encoded_column_names, index=df.index)

# Concatenate the original DataFrame with the encoded DataFrame
df = pd.concat([df, encoded_df], axis=1)

# Drop the original categorical columns
df.drop(columns=categorical_columns, inplace=True)

# Create lag features for the target variable
df['lag_speed_1'] = df['average_speed_kmh'].shift(1)
df['lag_speed_2'] = df['average_speed_kmh'].shift(2)

# Ensure proper temporal splitting by sorting data by time
df = df.sort_values(by=['collection_car', 'time'])

# Drop the first few rows where lag features would be NaN (after splitting the data)
df.dropna(inplace=True)

# Define features and target
X = df.drop(columns=['frame_id', 'country_code', 'collection_car', 'time', 'average_speed_kmh'])
y = df['average_speed_kmh']

# Split the data into train and test sets based on time (80% train, 20% test)
split_index = int(len(df) * 0.8)
X_train, X_test = X[:split_index], X[split_index:]
y_train, y_test = y[:split_index], y[split_index:]

# Initialize models
models = {
    'LinearRegression': LinearRegression(),
    'RandomForest': RandomForestRegressor(),
    'XGBoost': xgb.XGBRegressor(objective='reg:squarederror'),
    'NeuralNetwork': MLPRegressor(max_iter=1000),
    'Ridge': Ridge(),
    'Lasso': Lasso(),
    'ElasticNet': ElasticNet(),
    'GradientBoosting': GradientBoostingRegressor(),
    'AdaBoost': AdaBoostRegressor(),
    'ExtraTrees': ExtraTreesRegressor(),
    'KNeighbors': KNeighborsRegressor(),
    'SVR': SVR()
}

# Function to train and evaluate model
def train_and_evaluate_model(X_train, y_train, X_test, y_test, model):
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)

    mse = mean_squared_error(y_test, predictions)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, predictions)
    mae = mean_absolute_error(y_test, predictions)
    medae = median_absolute_error(y_test, predictions)
    return mse, rmse, r2, mae, medae, predictions

# Dictionary to store results
results = {}

# Loop through models and evaluate
for model_name, model in models.items():
    mse, rmse, r2, mae, medae, predictions = train_and_evaluate_model(X_train, y_train, X_test, y_test, model)
    results[model_name] = {'MSE': mse, 'RMSE': rmse, 'R2': r2, 'MAE': mae, 'MedAE': medae, 'Predictions': predictions}

# Displaying results
for model_name, metrics in results.items():
    print(f'Model: {model_name}, R2: {metrics["R2"]:.4f}, MAE: {metrics["MAE"]:.4f}, RMSE: {metrics["RMSE"]:.4f}, MedAE: {metrics["MedAE"]:.4f}')
    print('\n')



Model: LinearRegression, R2: 0.8658, MAE: 11.0925, RMSE: 15.0341, MedAE: 9.8835


Model: RandomForest, R2: 0.8944, MAE: 9.1030, RMSE: 13.3386, MedAE: 6.0900


Model: XGBoost, R2: 0.8130, MAE: 13.2255, RMSE: 17.7497, MedAE: 9.4878


Model: NeuralNetwork, R2: 0.8876, MAE: 9.6877, RMSE: 13.7629, MedAE: 7.7198


Model: Ridge, R2: 0.8659, MAE: 11.0840, RMSE: 15.0309, MedAE: 9.8769


Model: Lasso, R2: 0.8709, MAE: 9.8765, RMSE: 14.7458, MedAE: 7.7083


Model: ElasticNet, R2: 0.8644, MAE: 10.2908, RMSE: 15.1137, MedAE: 8.2804


Model: GradientBoosting, R2: 0.8622, MAE: 11.4402, RMSE: 15.2359, MedAE: 8.9659


Model: AdaBoost, R2: 0.8594, MAE: 12.0758, RMSE: 15.3875, MedAE: 11.9148


Model: ExtraTrees, R2: 0.8805, MAE: 9.9555, RMSE: 14.1890, MedAE: 7.0800


Model: KNeighbors, R2: 0.8657, MAE: 9.1593, RMSE: 15.0388, MedAE: 5.2000


Model: SVR, R2: 0.0181, MAE: 35.2284, RMSE: 40.6709, MedAE: 36.7940




In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error, median_absolute_error
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, GRU, Conv1D, MaxPooling1D, Flatten, Dropout
from tensorflow.keras.optimizers import Adam
from sklearn.preprocessing import MinMaxScaler

# Load the dataset
df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Final_Dataset_FR.csv')

# Convert time to datetime format
df['time'] = pd.to_datetime(df['time'], format='ISO8601')

# Extract temporal features from time column
df['hour'] = df['time'].dt.hour
df['day_of_week'] = df['time'].dt.dayofweek
df['day_of_month'] = df['time'].dt.day
df['month'] = df['time'].dt.month
df['year'] = df['time'].dt.year

# List of categorical columns to encode
categorical_columns = ['road_type', 'weather_condition', 'road_condition', 'time_of_day']

# One-hot encoding for categorical columns
df = pd.get_dummies(df, columns=categorical_columns, drop_first=True)

# Create lag features for the target variable
df['lag_speed_1'] = df['average_speed_kmh'].shift(1)
df['lag_speed_2'] = df['average_speed_kmh'].shift(2)

# Ensure proper temporal splitting by sorting data by time
df = df.sort_values(by=['collection_car', 'time'])

# Drop the first few rows where lag features would be NaN
df.dropna(inplace=True)

# Define features and target
X = df.drop(columns=['frame_id', 'country_code', 'collection_car', 'time', 'average_speed_kmh'])
y = df['average_speed_kmh']

# Normalize features
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

# Split the data into train and test sets based on time (80% train, 20% test)
split_index = int(len(df) * 0.8)
X_train, X_test = X_scaled[:split_index], X_scaled[split_index:]
y_train, y_test = y[:split_index], y[split_index:]

# Reshape data for LSTM/GRU which expect 3D input: (samples, timesteps, features)
X_train_reshaped = X_train.reshape(X_train.shape[0], 1, X_train.shape[1])
X_test_reshaped = X_test.reshape(X_test.shape[0], 1, X_test.shape[1])

# Function to train and evaluate deep learning model
def train_and_evaluate_dl_model(model, X_train, y_train, X_test, y_test):
    model.compile(optimizer=Adam(), loss='mse')
    model.fit(X_train, y_train, epochs=50, batch_size=32, verbose=0)
    predictions = model.predict(X_test)
    mse = mean_squared_error(y_test, predictions)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, predictions)
    mae = mean_absolute_error(y_test, predictions)
    medae = median_absolute_error(y_test, predictions)
    return mse, rmse, r2, mae, medae, predictions

# Initialize deep learning models
deep_learning_models = {
    'SimpleNN': Sequential([
        Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
        Dense(32, activation='relu'),
        Dense(1)
    ]),

    'LSTM': Sequential([
        LSTM(64, input_shape=(X_train_reshaped.shape[1], X_train_reshaped.shape[2])),
        Dense(32, activation='relu'),
        Dense(1)
    ]),

    'GRU': Sequential([
        GRU(64, input_shape=(X_train_reshaped.shape[1], X_train_reshaped.shape[2])),
        Dense(32, activation='relu'),
        Dense(1)
    ]),

    'Transformer': Sequential([
        Dense(128, activation='relu', input_shape=(X_train.shape[1],)),
        Dense(64, activation='relu'),
        Dense(32, activation='relu'),
        Dense(1)
    ])
}

# Dictionary to store results
results_dl = {}

# Train and evaluate each deep learning model
for model_name, model in deep_learning_models.items():
    if model_name in ['LSTM', 'GRU', 'CNN_LSTM']:
        mse, rmse, r2, mae, medae, predictions = train_and_evaluate_dl_model(model, X_train_reshaped, y_train, X_test_reshaped, y_test)
    else:
        mse, rmse, r2, mae, medae, predictions = train_and_evaluate_dl_model(model, X_train, y_train, X_test, y_test)

    results_dl[model_name] = {'MSE': mse, 'RMSE': rmse, 'R2': r2, 'MAE': mae, 'MedAE': medae, 'Predictions': predictions}

# Displaying results
for model_name, metrics in results_dl.items():
    print(f'Model: {model_name}, R2: {metrics["R2"]:.4f}, MAE: {metrics["MAE"]:.4f}, RMSE: {metrics["RMSE"]:.4f}, MedAE: {metrics["MedAE"]:.4f}\n')


54/54 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
54/54 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step
54/54 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step
54/54 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
Model: SimpleNN, R2: 0.8626, MAE: 10.4733, RMSE: 15.2138, MedAE: 6.8541

Model: LSTM, R2: 0.8684, MAE: 10.9757, RMSE: 14.8879, MedAE: 8.7762

Model: GRU, R2: 0.8766, MAE: 10.4238, RMSE: 14.4179, MedAE: 7.8241

Model: Transformer, R2: 0.7578, MAE: 12.8831, RMSE: 20.2004, MedAE: 6.8217



In [ ]:
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import load_model
import joblib
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# # Saving the best performing deep learning model (SimpleNN)
# best_dl_model = deep_learning_models['LSTM']
# best_dl_model.save('best_dl_model.keras')  # Save in the new Keras format

# Saving the best performing machine learning model (ExtraTrees)
best_ml_model = models['RandomForest']
joblib.dump(best_ml_model, 'best_ml_model.pkl')

joblib.dump(encoder, 'encoder.pkl')
joblib.dump(scaler, 'scaler.pkl')

# Preprocess input data
def preprocess_input(input_data, scaler, encoder, lag_1=None, lag_2=None):
    df_input = pd.DataFrame([input_data])

    # Convert time to datetime and extract temporal features
    df_input['time'] = pd.to_datetime(df_input['time'], format='ISO8601')
    df_input['hour'] = df_input['time'].dt.hour
    df_input['day_of_week'] = df_input['time'].dt.dayofweek
    df_input['day_of_month'] = df_input['time'].dt.day
    df_input['month'] = df_input['time'].dt.month
    df_input['year'] = df_input['time'].dt.year

    # Drop the time column
    df_input.drop(columns=['time'], inplace=True)

    # One-hot encode the categorical features (assuming encoder is already fitted)
    categorical_columns = ['road_type', 'weather_condition', 'road_condition', 'time_of_day']
    encoded_input = encoder.transform(df_input[categorical_columns])
    encoded_column_names = encoder.get_feature_names_out(categorical_columns)
    encoded_df = pd.DataFrame(encoded_input, columns=encoded_column_names)

    # Add lag features (previous speed values)
    df_input['lag_speed_1'] = lag_1
    df_input['lag_speed_2'] = lag_2

    # Drop original categorical columns and concatenate the one-hot encoded columns
    df_input.drop(columns=categorical_columns, inplace=True)
    df_input = pd.concat([df_input, encoded_df], axis=1)

    # Align columns in the same order as during model fitting
    feature_columns = scaler.feature_names_in_  # Ensure same order as during fitting
    df_input = df_input[feature_columns]

    # Scale the input using the scaler
    df_scaled = scaler.transform(df_input)

    return df_scaled

# Load the models
custom_objects = {'mse': tf.keras.losses.MeanSquaredError()}

# Load deep learning and machine learning models
#best_dl_model = load_model('best_dl_model.keras', custom_objects=custom_objects)
best_ml_model = joblib.load('best_ml_model.pkl')

# Predict with deep learning model
def predict_with_dl(input_data):
    # Preprocess input (assuming lag_1 and lag_2 are available)
    processed_input = preprocess_input(input_data, scaler, encoder, lag_1=input_data['lag_speed_1'], lag_2=input_data['lag_speed_2'])

    # Reshape for a feedforward NN, since SimpleNN is not LSTM/GRU
    processed_input_reshaped = processed_input.reshape(1,1, processed_input.shape[1])

    # Make prediction
    prediction = best_dl_model.predict(processed_input_reshaped)
    return prediction[0][0]

# Predict with machine learning model
def predict_with_ml(input_data):
    # Preprocess input
    processed_input = preprocess_input(input_data, scaler, encoder, lag_1=input_data['lag_speed_1'], lag_2=input_data['lag_speed_2'])

    # Make prediction
    prediction = best_ml_model.predict(processed_input)
    return prediction[0]

# Function to predict vehicle speed based on model type
def predict_vehicle_speed(input_data, model_type='SimpleNN'):
    if model_type == 'SimpleNN':
        return predict_with_dl(input_data)
    elif model_type == 'ExtraTrees':
        return predict_with_ml(input_data)
    else:
        raise ValueError(f"Model type {model_type} not supported.")

# Example input data
new_input = {
    'num_vehicles_ego_lane': 88,
    'num_vehicles_other_lane': 83,
    'num_vulnerable_vehicles_ego_lane': 80,
    'num_vulnerable_vehicles_other_lane': 80,
    'num_lane_instances': 0,
    'num_pedestrians_ego_lane': 80,
    'num_pedestrians_other_lane': 0,
    'num_animals_ego_lane': 0,
    'num_animals_other_lane': 0,
    'num_traffic_signs_ego_lane': 86,
    'num_traffic_signs_ego_road': 83,
    'num_traffic_signals_ego_lane': 85,
    'num_traffic_signals_ego_road': 86,
    'num_traffic_guides_ego_lane': 0,
    'num_traffic_guides_ego_road': 0,
    'num_traffic_beacons_ego_lane': 0,
    'num_traffic_beacons_ego_road': 0,
    'num_dynamic_barriers_ego_lane': 0,
    'num_dynamic_barriers_ego_road': 0,
    'ContainsCrossWalk': 1,
    'ContainsTrafficSign': 1,
    'ContainsMarker': 0,
    'ContainsOther': 0,
    'ContainsPictogram': 0,
    'ContainsText': 0,
    'time': '2024-10-23T12:00:00',
    'road_type': 'city',
    'weather_condition': 'rain',
    'road_condition': 'normal',
    'time_of_day': 'day',
    'lag_speed_1': 40.0,
    'lag_speed_2': 45.0,
}

# Predict using both models
speed_dl = predict_vehicle_speed(new_input, model_type='SimpleNN')
print(f"Predicted speed (DL): {speed_dl}")

speed_ml = predict_vehicle_speed(new_input, model_type='ExtraTrees')
print(f"Predicted speed (ML): {speed_ml}")


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 477ms/step
Predicted speed (DL): -0.22307686507701874
Predicted speed (ML): 15.21


In [9]:
import pandas as pd
import joblib
import time
import psutil
import os

# Reload the Random Forest model, encoder, and scaler
best_ml_model = joblib.load('/content/drive/MyDrive/best_ml_model.pkl')
encoder = joblib.load('/content/drive/MyDrive/encoder.pkl')
scaler = joblib.load('/content/drive/MyDrive/scaler.pkl')

# Preprocess input data
def preprocess_input(input_data, scaler, encoder, lag_1=None, lag_2=None):
    df_input = pd.DataFrame([input_data])

    # Convert time to datetime and extract temporal features
    df_input['time'] = pd.to_datetime(df_input['time'], format='ISO8601')
    df_input['hour'] = df_input['time'].dt.hour
    df_input['day_of_week'] = df_input['time'].dt.dayofweek
    df_input['day_of_month'] = df_input['time'].dt.day
    df_input['month'] = df_input['time'].dt.month
    df_input['year'] = df_input['time'].dt.year

    # Drop the time column
    df_input.drop(columns=['time'], inplace=True)

    # One-hot encode the categorical features
    categorical_columns = ['road_type', 'weather_condition', 'road_condition', 'time_of_day']
    encoded_input = encoder.transform(df_input[categorical_columns])
    encoded_column_names = encoder.get_feature_names_out(categorical_columns)
    encoded_df = pd.DataFrame(encoded_input, columns=encoded_column_names)

    # Add lag features
    df_input['lag_speed_1'] = lag_1
    df_input['lag_speed_2'] = lag_2

    # Drop original categorical columns and concatenate the one-hot encoded columns
    df_input.drop(columns=categorical_columns, inplace=True)
    df_input = pd.concat([df_input, encoded_df], axis=1)

    # Align columns in the same order as during model fitting
    feature_columns = scaler.feature_names_in_  # Ensure same order as during fitting
    df_input = df_input[feature_columns]

    # Scale the input using the scaler
    df_scaled = scaler.transform(df_input)

    return df_scaled

# Predict with Random Forest model and measure time
def predict_with_ml(input_data):
    # Monitor CPU and memory usage before prediction
    process = psutil.Process(os.getpid())
    mem_before = process.memory_info().rss / 1024 ** 2  # Memory in MB
    cpu_before = psutil.cpu_percent(interval=None)

    # Start timing the prediction
    start_time = time.time()

    # Preprocess input
    processed_input = preprocess_input(
        input_data,
        scaler,
        encoder,
        lag_1=input_data['lag_speed_1'],
        lag_2=input_data['lag_speed_2']
    )

    # Make prediction
    prediction = best_ml_model.predict(processed_input)

    # End timing
    end_time = time.time()
    elapsed_time = end_time - start_time  # Time in seconds

    # Monitor CPU and memory usage after prediction
    mem_after = process.memory_info().rss / 1024 ** 2  # Memory in MB
    cpu_after = psutil.cpu_percent(interval=None)

    # Calculate memory and CPU utilization during prediction
    mem_used = mem_after - mem_before
    cpu_used = cpu_after - cpu_before

    return prediction[0], elapsed_time, mem_used, cpu_used

# Example input data
new_input = {
    'num_vehicles_ego_lane': 110,
    'num_vehicles_other_lane': 0,
    'num_vulnerable_vehicles_ego_lane': 0,
    'num_vulnerable_vehicles_other_lane': 0,
    'num_lane_instances': 0,
    'num_pedestrians_ego_lane': 0,
    'num_pedestrians_other_lane': 0,
    'num_animals_ego_lane': 0,
    'num_animals_other_lane': 0,
    'num_traffic_signs_ego_lane': 0,
    'num_traffic_signs_ego_road': 0,
    'num_traffic_signals_ego_lane': 0,
    'num_traffic_signals_ego_road': 0,
    'num_traffic_guides_ego_lane': 0,
    'num_traffic_guides_ego_road': 0,
    'num_traffic_beacons_ego_lane': 0,
    'num_traffic_beacons_ego_road': 0,
    'num_dynamic_barriers_ego_lane': 0,
    'num_dynamic_barriers_ego_road': 0,
    'ContainsCrossWalk': 0,
    'ContainsTrafficSign': 0,
    'ContainsMarker': 0,
    'ContainsOther': 0,
    'ContainsPictogram': 0,
    'ContainsText': 0,
    'time': '2024-10-23T12:00:00',
    'road_type': 'city',
    'weather_condition': 'rain',
    'road_condition': 'normal',
    'time_of_day': 'day',
    'lag_speed_1': 50.0,
    'lag_speed_2': 100.0,
}

# Predict and get resource utilization
speed_ml, prediction_time, memory_used, cpu_used = predict_with_ml(new_input)

# Print results
print(f"Predicted speed (ML): {speed_ml}")
print(f"Prediction time: {prediction_time:.4f} seconds")
print(f"Memory used: {memory_used:.2f} MB")
print(f"CPU used: {cpu_used:.2f}%")


Predicted speed (ML): 7.27
Prediction time: 0.0343 seconds
Memory used: 0.00 MB
CPU used: 60.30%


/usr/local/lib/python3.10/dist-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
